In [ ]:
import pandas as pd
import sqlite3

db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

print("📥 1. Loading raw tables into memory (Taking only what we need)...")
demo = pd.read_sql("SELECT primaryid, age, wt, sex, occp_cod, rept_cod, is_test_set FROM demo_clean WHERE age IS NOT NULL", conn)
reac = pd.read_sql("SELECT primaryid, pt FROM reac_clean WHERE pt IS NOT NULL", conn)
drug = pd.read_sql("SELECT primaryid, final_drug_name, role_cod, route FROM drug_clean", conn)
ther = pd.read_sql("SELECT primaryid, dur FROM ther_clean WHERE dur IS NOT NULL", conn)
rpsr = pd.read_sql("SELECT primaryid, rpsr_cod FROM rpsr_clean", conn)
indi = pd.read_sql("SELECT primaryid, indi_pt FROM indi_clean WHERE indi_pt IS NOT NULL", conn)

print("⚙️ 2. Processing Reaction (The Window Function equivalent)...")
# ده بديل سطر ROW_NUMBER() OVER(PARTITION BY) وبياخد ثواني في بانداز
reac_target = reac.sort_values(by='pt').drop_duplicates(subset=['primaryid'], keep='first')
reac_target.rename(columns={'pt': 'target_reaction'}, inplace=True)

print("💊 3. Processing Drugs (Primary Suspect & Polypharmacy)...")

# 🔥 التعديل هنا: تحويل القيم لنصوص صريحة ومعالجة الـ NaN عشان Pandas ميهنجش
drug['final_drug_name'] = drug['final_drug_name'].astype(str)
drug['route'] = drug['route'].fillna('Unknown').astype(str)

drug_ps = drug[(drug['role_cod'] == 'PS') & (drug['final_drug_name'] != 'none')]
drug_primary = drug_ps.groupby('primaryid').agg(
    primary_suspect_drug=('final_drug_name', 'max'),
    ps_route=('route', 'max')
).reset_index()

polypharmacy = drug.groupby('primaryid').size().reset_index(name='num_drugs')

print("⏱️ 4. Processing Therapy, Source, and Indications...")
therapy = ther.groupby('primaryid').agg(therapy_duration=('dur', 'max')).reset_index()
report_src = rpsr.groupby('primaryid').agg(rpsr_cod=('rpsr_cod', 'max')).reset_index()

indications = indi.groupby('primaryid').agg(
    num_indications=('indi_pt', 'size'),
    primary_indication=('indi_pt', 'max')
).reset_index()

print("🔗 5. Merging everything together (The BIG JOIN)...")
# الـ INNER JOINS الأساسية
df_matrix_b = demo.merge(drug_primary, on='primaryid', how='inner')
df_matrix_b = df_matrix_b.merge(reac_target, on='primaryid', how='inner')

# الـ LEFT JOINS
df_matrix_b = df_matrix_b.merge(polypharmacy, on='primaryid', how='left')
df_matrix_b['num_drugs'] = df_matrix_b['num_drugs'].fillna(1)

df_matrix_b = df_matrix_b.merge(therapy, on='primaryid', how='left')
df_matrix_b = df_matrix_b.merge(report_src, on='primaryid', how='left')

df_matrix_b = df_matrix_b.merge(indications, on='primaryid', how='left')
df_matrix_b['num_indications'] = df_matrix_b['num_indications'].fillna(0)

print(f"✅ Matrix B built successfully in Pandas! Final Shape: {df_matrix_b.shape}")

print("💾 6. Saving final matrix back to SQLite for future use...")
df_matrix_b.to_sql('matrix_b_reaction_final', conn, if_exists='replace', index=False)
print("🎉 Done! You can now run the Ensemble Machine Learning code.")

In [ ]:
import pandas as pd
import sqlite3
import numpy as np
import joblib
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.utils import resample, shuffle

db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

print("🚀 Loading Matrix B for Multi-Class Ensemble...")
df_ml = pd.read_sql_query("SELECT * FROM matrix_b_reaction_final", conn)

# ---------------------------------------------------------
# 1. Target Hybrid Mapping (Top 20 + Other)
# ---------------------------------------------------------
print("⚙️ Step 1: Target Hybrid Mapping...")
top_20_reactions = df_ml['target_reaction'].value_counts().nlargest(20).index.tolist()
df_ml['target_mapped'] = df_ml['target_reaction'].apply(lambda x: x if x in top_20_reactions else 'Other')

le_target = LabelEncoder()
df_ml['target_encoded'] = le_target.fit_transform(df_ml['target_mapped'])
num_classes = len(le_target.classes_)
print(f"📊 Number of Classes: {num_classes}")

# ---------------------------------------------------------
# 2. Time-Based Split (Out-of-Time Validation)
# ---------------------------------------------------------
print("\n✂️ Step 2: Out-of-Time Validation Split...")
train_mask = df_ml['is_test_set'] == 0
test_mask = df_ml['is_test_set'] == 1

drop_cols = ['target_reaction', 'target_mapped', 'target_encoded', 'is_test_set', 'primaryid']
X_train = df_ml[train_mask].drop(columns=drop_cols).copy()
y_train = df_ml[train_mask]['target_encoded'].copy()

X_test = df_ml[test_mask].drop(columns=drop_cols).copy()
y_test = df_ml[test_mask]['target_encoded'].copy()

# ---------------------------------------------------------
# 2.5 The Fix: Balancing Data (Under-sampling 'Other')
# ---------------------------------------------------------
print("\n⚖️ Step 2.5: Balancing Data (Under-sampling the 'Other' class)...")
other_class_val = le_target.transform(['Other'])[0]

X_majority = X_train[y_train == other_class_val]
y_majority = y_train[y_train == other_class_val]

X_minority = X_train[y_train != other_class_val]
y_minority = y_train[y_train != other_class_val]

# بنقص كلاس Other عشان يتساوى مع مجموع الـ 20 كلاس التانيين
X_majority_downsampled, y_majority_downsampled = resample(
    X_majority, 
    y_majority,
    replace=False,
    n_samples=len(X_minority), 
    random_state=42
)

X_train = pd.concat([X_majority_downsampled, X_minority])
y_train = pd.concat([y_majority_downsampled, y_minority])

X_train, y_train = shuffle(X_train, y_train, random_state=42)
print(f"📦 NEW Balanced Train Shape: {X_train.shape} | Test Shape remains: {X_test.shape}")

# ---------------------------------------------------------
# 3. Categorical Handling (Ordinal Encoding)
# ---------------------------------------------------------
print("\n🧹 Step 3: Ordinal Encoding & Handling NaNs...")
cat_cols = ['primary_suspect_drug', 'ps_route', 'rept_cod', 'occp_cod', 'rpsr_cod', 'primary_indication', 'sex']
num_cols = ['age', 'wt', 'num_drugs', 'therapy_duration', 'num_indications']

X_train[cat_cols] = X_train[cat_cols].fillna('Unknown').astype(str)
X_test[cat_cols] = X_test[cat_cols].fillna('Unknown').astype(str)
X_train[num_cols] = X_train[num_cols].fillna(-1)
X_test[num_cols] = X_test[num_cols].fillna(-1)

# handle_unknown='use_encoded_value' بيحمي الموديل لو ظهر دواء جديد في المستقبل
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = oe.fit_transform(X_train[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])

# ---------------------------------------------------------
# 4. Initializing Models
# ---------------------------------------------------------
print("\n🤖 Step 4: Initializing Models (Pure logic, no artificial weights)...")
xgb_clf = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=num_classes,
    max_depth=7,
    learning_rate=0.1,
    n_estimators=300,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

lgb_clf = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=num_classes,
    max_depth=7,
    learning_rate=0.1,
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

# ---------------------------------------------------------
# 5. Training Ensemble
# ---------------------------------------------------------
print("\n🤝 Step 5: Training The Ultimate Soft Voting Ensemble...")
ensemble_model = VotingClassifier(
    estimators=[('XGB', xgb_clf), ('LGBM', lgb_clf), ('RF', rf_clf)],
    voting='soft',
    n_jobs=1 
)

ensemble_model.fit(X_train, y_train)

# ---------------------------------------------------------
# 6. Final Evaluation
# ---------------------------------------------------------
print("\n🎯 Step 6: Evaluating Ensemble on Future Data (Q4)...")
y_pred = ensemble_model.predict(X_test)

print(f"🌟 ENSEMBLE ACCURACY: {accuracy_score(y_test, y_pred):.4f}")
print(f"🌟 ENSEMBLE F1-SCORE (MACRO): {f1_score(y_test, y_pred, average='macro'):.4f}\n")

# ---------------------------------------------------------
# 7. Saving the Architecture
# ---------------------------------------------------------
print("💾 Step 7: Saving Models and Encoders to disk...")
joblib.dump(ensemble_model, '../models/matrix_b_ensemble.pkl')
joblib.dump(le_target, '../models/le_matrix_b.pkl')
joblib.dump(oe, '../models/oe_matrix_b.pkl')
print("✅ Pipeline Completed and Saved!")

In [ ]:
import pandas as pd
import sqlite3
import numpy as np
import joblib
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.utils import resample, shuffle

db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

print("🚀 Loading Matrix B for EasyEnsemble...")
df_ml = pd.read_sql_query("SELECT * FROM matrix_b_reaction_final", conn)

# ---------------------------------------------------------
# 1. Target Hybrid Mapping
# ---------------------------------------------------------
print("⚙️ Step 1: Target Hybrid Mapping...")
top_20_reactions = df_ml['target_reaction'].value_counts().nlargest(20).index.tolist()
df_ml['target_mapped'] = df_ml['target_reaction'].apply(lambda x: x if x in top_20_reactions else 'Other')

le_target = LabelEncoder()
df_ml['target_encoded'] = le_target.fit_transform(df_ml['target_mapped'])
num_classes = len(le_target.classes_)

# ---------------------------------------------------------
# 2. Time-Based Split
# ---------------------------------------------------------
print("✂️ Step 2: Out-of-Time Validation Split...")
train_mask = df_ml['is_test_set'] == 0
test_mask = df_ml['is_test_set'] == 1

drop_cols = ['target_reaction', 'target_mapped', 'target_encoded', 'is_test_set', 'primaryid']
X_train = df_ml[train_mask].drop(columns=drop_cols).copy()
y_train = df_ml[train_mask]['target_encoded'].copy()

X_test = df_ml[test_mask].drop(columns=drop_cols).copy()
y_test = df_ml[test_mask]['target_encoded'].copy()

# ---------------------------------------------------------
# 3. Ordinal Encoding
# ---------------------------------------------------------
print("🧹 Step 3: Ordinal Encoding & Handling NaNs...")
cat_cols = ['primary_suspect_drug', 'ps_route', 'rept_cod', 'occp_cod', 'rpsr_cod', 'primary_indication', 'sex']
num_cols = ['age', 'wt', 'num_drugs', 'therapy_duration', 'num_indications']

X_train[cat_cols] = X_train[cat_cols].fillna('Unknown').astype(str)
X_test[cat_cols] = X_test[cat_cols].fillna('Unknown').astype(str)
X_train[num_cols] = X_train[num_cols].fillna(-1)
X_test[num_cols] = X_test[num_cols].fillna(-1)

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = oe.fit_transform(X_train[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])

# ---------------------------------------------------------
# 4. EasyEnsemble Magic Loop 🪄
# ---------------------------------------------------------
print("\n🔥 Step 4: Initiating EasyEnsemble Training (3 Iterations)...")
other_class_val = le_target.transform(['Other'])[0]

X_majority = X_train[y_train == other_class_val]
y_majority = y_train[y_train == other_class_val]
X_minority = X_train[y_train != other_class_val]
y_minority = y_train[y_train != other_class_val]

n_ensembles = 3 # عدد الموديلات اللي هتتدرب وتصوت في النهاية
trained_easy_ensembles = []

for i in range(n_ensembles):
    print(f"\n🌀 Training Sub-Ensemble [{i+1}/{n_ensembles}]...")
    
    # 1. سحب عينة عشوائية مختلفة من كلاس Other في كل لفة (باستخدام random_state متغير)
    X_maj_down, y_maj_down = resample(
        X_majority, y_majority, replace=False,
        n_samples=len(X_minority), random_state=42 + i 
    )
    
    # 2. دمج العينة مع الكلاسات النادرة
    X_train_chunk = pd.concat([X_maj_down, X_minority])
    y_train_chunk = pd.concat([y_maj_down, y_minority])
    X_train_chunk, y_train_chunk = shuffle(X_train_chunk, y_train_chunk, random_state=42)
    
    # 3. تهيئة الموديلات الأساسية
    xgb_clf = xgb.XGBClassifier(objective='multi:softprob', num_class=num_classes, max_depth=7, learning_rate=0.1, n_estimators=300, tree_method='hist', random_state=42, n_jobs=-1)
    lgb_clf = lgb.LGBMClassifier(objective='multiclass', num_class=num_classes, max_depth=7, learning_rate=0.1, n_estimators=300, random_state=42, n_jobs=-1, verbose=-1) # verbose=-1 عشان نمنع رسايل الـ splits
    rf_clf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=10, random_state=42, n_jobs=-1)
    
    # 4. التدريب
    ensemble_model = VotingClassifier(estimators=[('XGB', xgb_clf), ('LGBM', lgb_clf), ('RF', rf_clf)], voting='soft', n_jobs=1)
    ensemble_model.fit(X_train_chunk, y_train_chunk)
    
    # حفظ الموديل المتدرب في القائمة
    trained_easy_ensembles.append(ensemble_model)

# ---------------------------------------------------------
# 5. EasyEnsemble Evaluation (Voting among Ensembles)
# ---------------------------------------------------------
print("\n🎯 Step 5: Evaluating The Mega-Ensemble on Future Data (Q4)...")
# تجميع الاحتمالات (Probabilities) من الـ 3 موديلات وأخد المتوسط بتاعهم
y_pred_proba = np.zeros((len(X_test), num_classes))

for model in trained_easy_ensembles:
    y_pred_proba += model.predict_proba(X_test)
    
y_pred_proba /= n_ensembles
y_pred = np.argmax(y_pred_proba, axis=1)

print(f"🌟 MEGA-ENSEMBLE ACCURACY: {accuracy_score(y_test, y_pred):.4f}")
print(f"🌟 MEGA-ENSEMBLE F1-SCORE (MACRO): {f1_score(y_test, y_pred, average='macro'):.4f}\n")

# ---------------------------------------------------------
# 6. Saving
# ---------------------------------------------------------
print("💾 Step 6: Saving the List of Ensembles to disk...")
# هنحفظ الـ List كلها في ملف واحد عشان الداشبورد تعرف تقراه
joblib.dump(trained_easy_ensembles, '../models/matrix_b_easy_ensemble.pkl')
joblib.dump(le_target, '../models/le_matrix_b.pkl')
joblib.dump(oe, '../models/oe_matrix_b.pkl')
print("✅ EasyEnsemble Pipeline Completed and Saved!")

In [ ]:
import pandas as pd
import sqlite3
import numpy as np
import joblib
import xgboost as xgb
import lightgbm as lgb
import optuna
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.utils import resample, shuffle

# ---------------------------------------------------------
# 1. Loading & Preparing Data (Same robust logic)
# ---------------------------------------------------------
print("🚀 Loading Data for OPTUNA Hyperparameter Tuning...")
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)
df_ml = pd.read_sql_query("SELECT * FROM matrix_b_reaction_final", conn)

top_20_reactions = df_ml['target_reaction'].value_counts().nlargest(20).index.tolist()
df_ml['target_mapped'] = df_ml['target_reaction'].apply(lambda x: x if x in top_20_reactions else 'Other')

le_target = LabelEncoder()
df_ml['target_encoded'] = le_target.fit_transform(df_ml['target_mapped'])
num_classes = len(le_target.classes_)

train_mask = df_ml['is_test_set'] == 0
test_mask = df_ml['is_test_set'] == 1

drop_cols = ['target_reaction', 'target_mapped', 'target_encoded', 'is_test_set', 'primaryid']
X_train = df_ml[train_mask].drop(columns=drop_cols).copy()
y_train = df_ml[train_mask]['target_encoded'].copy()
X_test = df_ml[test_mask].drop(columns=drop_cols).copy()
y_test = df_ml[test_mask]['target_encoded'].copy()

# Balancing Data (Under-sampling)
other_class_val = le_target.transform(['Other'])[0]
X_majority = X_train[y_train == other_class_val]
y_majority = y_train[y_train == other_class_val]
X_minority = X_train[y_train != other_class_val]
y_minority = y_train[y_train != other_class_val]

X_maj_down, y_maj_down = resample(X_majority, y_majority, replace=False, n_samples=len(X_minority), random_state=42)
X_train = pd.concat([X_maj_down, X_minority])
y_train = pd.concat([y_maj_down, y_minority])
X_train, y_train = shuffle(X_train, y_train, random_state=42)

# Ordinal Encoding
cat_cols = ['primary_suspect_drug', 'ps_route', 'rept_cod', 'occp_cod', 'rpsr_cod', 'primary_indication', 'sex']
num_cols = ['age', 'wt', 'num_drugs', 'therapy_duration', 'num_indications']

X_train[cat_cols] = X_train[cat_cols].fillna('Unknown').astype(str)
X_test[cat_cols] = X_test[cat_cols].fillna('Unknown').astype(str)
X_train[num_cols] = X_train[num_cols].fillna(-1)
X_test[num_cols] = X_test[num_cols].fillna(-1)

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = oe.fit_transform(X_train[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])


# ---------------------------------------------------------
# 2. OPTUNA Objective Function 🎯
# ---------------------------------------------------------
def objective(trial):
    # 1. تحديد الـ Search Space للـ Hyperparameters
    # XGBoost Parameters
    xgb_depth = trial.suggest_int('xgb_max_depth', 5, 12)
    xgb_lr = trial.suggest_float('xgb_learning_rate', 0.01, 0.2, log=True)
    xgb_n_est = trial.suggest_int('xgb_n_estimators', 100, 400, step=50)
    
    # LightGBM Parameters
    lgb_depth = trial.suggest_int('lgb_max_depth', 5, 12)
    lgb_lr = trial.suggest_float('lgb_learning_rate', 0.01, 0.2, log=True)
    lgb_n_est = trial.suggest_int('lgb_n_estimators', 100, 400, step=50)
    
    # 2. بناء الموديلات بالقيم الجديدة
    xgb_clf = xgb.XGBClassifier(
        objective='multi:softprob', num_class=num_classes, tree_method='hist',
        max_depth=xgb_depth, learning_rate=xgb_lr, n_estimators=xgb_n_est,
        random_state=42, n_jobs=-1
    )
    
    lgb_clf = lgb.LGBMClassifier(
        objective='multiclass', num_class=num_classes, verbose=-1,
        max_depth=lgb_depth, learning_rate=lgb_lr, n_estimators=lgb_n_est,
        random_state=42, n_jobs=-1
    )
    
    # Random Forest هنثبته عشان منستهلكش وقت خرافي
    rf_clf = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
    
    # 3. تجميعهم في Ensemble وتدريبهم
    ensemble = VotingClassifier(
        estimators=[('XGB', xgb_clf), ('LGBM', lgb_clf), ('RF', rf_clf)],
        voting='soft', n_jobs=1
    )
    
    ensemble.fit(X_train, y_train)
    
    # 4. التقييم بالـ F1 Macro
    y_pred = ensemble.predict(X_test)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    
    return f1_macro

# ---------------------------------------------------------
# 3. Running the Optimization Study 🏃‍♂️
# ---------------------------------------------------------
print("\n🔥 Starting Optuna Hyperparameter Tuning...")
print("⚠️ This will take some time. Go grab a coffee! ☕")

# إحنا بندور على أعلى F1-Score، فهنخلي الاتجاه Maximize
study = optuna.create_study(direction='maximize', study_name="Oriva_Matrix_B_Tuning")

# هنجرب 10 محاولات بس كبداية (ممكن تزودهم لـ 30 بعدين لو اللاب توب مستحمل)
study.optimize(objective, n_trials=10)

# ---------------------------------------------------------
# 4. Results & Saving the Best Final Model 🏆
# ---------------------------------------------------------
print("\n🎉 OPTUNA TUNING COMPLETED!")
print("Best F1-Score Achieved:", study.best_value)
print("Best Parameters Discovered:", study.best_params)

print("\n🚀 Training Final Model with Best Parameters...")
best_xgb = xgb.XGBClassifier(
    objective='multi:softprob', num_class=num_classes, tree_method='hist',
    max_depth=study.best_params['xgb_max_depth'], 
    learning_rate=study.best_params['xgb_learning_rate'], 
    n_estimators=study.best_params['xgb_n_estimators'],
    random_state=42, n_jobs=-1
)

best_lgb = lgb.LGBMClassifier(
    objective='multiclass', num_class=num_classes, verbose=-1,
    max_depth=study.best_params['lgb_max_depth'], 
    learning_rate=study.best_params['lgb_learning_rate'], 
    n_estimators=study.best_params['lgb_n_estimators'],
    random_state=42, n_jobs=-1
)

best_rf = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)

final_ensemble = VotingClassifier(
    estimators=[('XGB', best_xgb), ('LGBM', best_lgb), ('RF', best_rf)],
    voting='soft', n_jobs=1
)

final_ensemble.fit(X_train, y_train)
final_preds = final_ensemble.predict(X_test)

print(f"\n🌟 FINAL TUNED ACCURACY: {accuracy_score(y_test, final_preds):.4f}")
print(f"🌟 FINAL TUNED F1-SCORE (MACRO): {f1_score(y_test, final_preds, average='macro'):.4f}")

# Save Everything
joblib.dump(final_ensemble, '../models/matrix_b_ensemble_TUNED.pkl')
joblib.dump(le_target, '../models/le_matrix_b.pkl')
joblib.dump(oe, '../models/oe_matrix_b.pkl')
print("\n✅ Final Tuned Pipeline Saved Successfully!")